In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
train = pd.read_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/raw/train.csv')
test = pd.read_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/raw/test.csv')

Mounted at /content/drive


In [2]:
print(train.info())
print(train.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [3]:
import pandas as pd
import numpy as np

# Load and combine
train = pd.read_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/raw/train.csv')
test = pd.read_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/raw/test.csv')
df = pd.concat([train, test], ignore_index=True)

print(f"Combined rows: {len(df)}")

# Drop the two identifier columns — neither is useful for analysis
df = df.drop(columns=['Unnamed: 0', 'id'])

# Standardise inconsistent capitalisation
df['Customer Type'] = df['Customer Type'].replace({
    'disloyal Customer': 'Disloyal Customer'
})

# The 14 service-rating columns
touchpoints = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Report how many 0s exist per touchpoint before we touch them
zero_counts = (df[touchpoints] == 0).sum().sort_values(ascending=False)
print("\nRatings of 0 per touchpoint (to be treated as missing):")
print(zero_counts)

# Treat 0 as missing for each touchpoint individually — replace with NaN
# This keeps the row, just excludes that one column's 0 from that column's calculations
df[touchpoints] = df[touchpoints].replace(0, np.nan)

# Missing arrival delay — just report it for now, decide handling once we see the scale
print(f"\nMissing Arrival Delay in Minutes: {df['Arrival Delay in Minutes'].isna().sum()}")

print(f"\nFinal combined shape: {df.shape}")
df.head()

Combined rows: 129880

Ratings of 0 per touchpoint (to be treated as missing):
Departure/Arrival time convenient    6681
Ease of Online booking               5682
Inflight wifi service                3916
Online boarding                      3080
Leg room service                      598
Food and drink                        132
Inflight entertainment                 18
Cleanliness                            14
Inflight service                        5
On-board service                        5
Gate location                           1
Seat comfort                            1
Checkin service                         1
Baggage handling                        0
dtype: int64

Missing Arrival Delay in Minutes: 393

Final combined shape: (129880, 23)


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3.0,4.0,3.0,1.0,...,5.0,4.0,3.0,4,4.0,5.0,5.0,25,18.0,neutral or dissatisfied
1,Male,Disloyal Customer,25,Business travel,Business,235,3.0,2.0,3.0,3.0,...,1.0,1.0,5.0,3,1.0,4.0,1.0,1,6.0,neutral or dissatisfied
2,Female,Loyal Customer,26,Business travel,Business,1142,2.0,2.0,2.0,2.0,...,5.0,4.0,3.0,4,4.0,4.0,5.0,0,0.0,satisfied
3,Female,Loyal Customer,25,Business travel,Business,562,2.0,5.0,5.0,5.0,...,2.0,2.0,5.0,3,1.0,4.0,2.0,11,9.0,neutral or dissatisfied
4,Male,Loyal Customer,61,Business travel,Business,214,3.0,3.0,3.0,3.0,...,3.0,3.0,4.0,4,3.0,3.0,3.0,0,0.0,satisfied


In [4]:
before = len(df)
df = df.dropna(subset=['Arrival Delay in Minutes'])
print(f"Dropped {before - len(df)} rows for missing Arrival Delay in Minutes")
print(f"Remaining rows: {len(df)}")

Dropped 393 rows for missing Arrival Delay in Minutes
Remaining rows: 129487


In [5]:
top2box = {}
for col in touchpoints:
    valid = df[col].dropna()
    top2box[col] = (valid.isin([4, 5])).mean() * 100

top2box_df = pd.DataFrame.from_dict(top2box, orient='index', columns=['Top-2-Box %'])
top2box_df = top2box_df.sort_values('Top-2-Box %', ascending=False).round(1)
print(top2box_df)

                                   Top-2-Box %
Inflight service                          62.7
Baggage handling                          62.1
Seat comfort                              56.2
Inflight entertainment                    52.6
On-board service                          52.5
Leg room service                          51.7
Online boarding                           50.9
Departure/Arrival time convenient         48.6
Cleanliness                               48.0
Checkin service                           47.9
Food and drink                            45.1
Gate location                             36.9
Ease of Online booking                    33.7
Inflight wifi service                     31.1


In [6]:
top2box_full = {}
for col in touchpoints:
    valid = df[col].dropna()
    top2box_full[col] = {
        'Top-2-Box %': round((valid.isin([4, 5])).mean() * 100, 1),
        'N (rated)': len(valid)
    }

top2box_full_df = pd.DataFrame.from_dict(top2box_full, orient='index')
top2box_full_df = top2box_full_df.sort_values('Top-2-Box %', ascending=False)
print(top2box_full_df)

                                   Top-2-Box %  N (rated)
Inflight service                          62.7     129482
Baggage handling                          62.1     129487
Seat comfort                              56.2     129486
Inflight entertainment                    52.6     129469
On-board service                          52.5     129482
Leg room service                          51.7     128891
Online boarding                           50.9     126416
Departure/Arrival time convenient         48.6     122823
Cleanliness                               48.0     129473
Checkin service                           47.9     129486
Food and drink                            45.1     129357
Gate location                             36.9     129486
Ease of Online booking                    33.7     123821
Inflight wifi service                     31.1     125579


In [7]:
import statsmodels.api as sm
import pandas as pd

# Encode the target
df['satisfied_binary'] = (df['satisfaction'] == 'satisfied').astype(int)

# Build the model dataframe: touchpoints + delays + controls
model_df = df[touchpoints + ['Departure Delay in Minutes', 'Arrival Delay in Minutes',
                               'Class', 'Type of Travel', 'Customer Type',
                               'satisfied_binary']].dropna()

# One-hot encode the categorical controls (drop_first avoids the dummy trap)
model_df = pd.get_dummies(model_df, columns=['Class', 'Type of Travel', 'Customer Type'],
                            drop_first=True)

print(f"Rows going into the model: {len(model_df)}")

X = model_df.drop(columns=['satisfied_binary'])
y = model_df['satisfied_binary']

# statsmodels wants floats and an explicit constant
X = X.astype(float)
X = sm.add_constant(X)

logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())

Rows going into the model: 119204
Optimization terminated successfully.
         Current function value: 0.245321
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:       satisfied_binary   No. Observations:               119204
Model:                          Logit   Df Residuals:                   119183
Method:                           MLE   Df Model:                           20
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                  0.6405
Time:                        13:02:46   Log-Likelihood:                -29243.
converged:                       True   LL-Null:                       -81343.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                       

In [8]:
# Check if these two touchpoints correlate oddly with travel type / class
check_cols = ['Departure/Arrival time convenient', 'Gate location']
for col in check_cols:
    print(f"\n{col} — mean rating by group:")
    print(df.groupby('Type of Travel')[col].mean())
    print(df.groupby('Class')[col].mean())


Departure/Arrival time convenient — mean rating by group:
Type of Travel
Business travel    2.981287
Personal Travel    3.743657
Name: Departure/Arrival time convenient, dtype: float64
Class
Business    3.047288
Eco         3.398330
Eco Plus    3.316514
Name: Departure/Arrival time convenient, dtype: float64

Gate location — mean rating by group:
Type of Travel
Business travel    3.002504
Personal Travel    2.919809
Name: Gate location, dtype: float64
Class
Business    2.985030
Eco         2.969699
Eco Plus    2.968230
Name: Gate location, dtype: float64


In [9]:
marginal_effects = logit_model.get_margeff(at='overall')
print(marginal_effects.summary())

        Logit Marginal Effects       
Dep. Variable:       satisfied_binary
Method:                          dydx
At:                           overall
                                       dy/dx    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Inflight wifi service                 0.0576      0.001     63.902      0.000       0.056       0.059
Departure/Arrival time convenient    -0.0253      0.001    -28.533      0.000      -0.027      -0.024
Ease of Online booking                0.0285      0.001     28.057      0.000       0.027       0.030
Gate location                        -0.0191      0.001    -21.724      0.000      -0.021      -0.017
Food and drink                       -0.0035      0.001     -4.154      0.000      -0.005      -0.002
Online boarding                       0.0675      0.001     81.999      0.000       0.066       0.069
Seat comfort                    

In [10]:
# 1. Multicollinearity — VIF check
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X.drop(columns=['const'])
vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print("=== VIF (multicollinearity check) ===")
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))

# 2. Outcome class balance
print("\n=== Satisfaction split (full data) ===")
print(df['satisfaction'].value_counts(normalize=True))

# 3. Whether dropped rows differ systematically from kept rows
dropped_rows = df[df[touchpoints].isna().any(axis=1)]
print(f"\n=== Dropped rows: {len(dropped_rows)} ({len(dropped_rows)/len(df)*100:.1f}% of data) ===")
print("\nSatisfaction split — dropped rows:")
print(dropped_rows['satisfaction'].value_counts(normalize=True))
print("\nSatisfaction split — full data (for comparison):")
print(df['satisfaction'].value_counts(normalize=True))
print("\nClass split — dropped rows:")
print(dropped_rows['Class'].value_counts(normalize=True))
print("\nClass split — full data (for comparison):")
print(df['Class'].value_counts(normalize=True))
print("\nType of Travel split — dropped rows:")
print(dropped_rows['Type of Travel'].value_counts(normalize=True))
print("\nType of Travel split — full data (for comparison):")
print(df['Type of Travel'].value_counts(normalize=True))

# 4. Outlier check on Age and Flight Distance
print("\n=== Age ===")
print(df['Age'].describe())
print("\n=== Flight Distance ===")
print(df['Flight Distance'].describe())

=== VIF (multicollinearity check) ===
                          feature       VIF
         Arrival Delay in Minutes 14.580098
       Departure Delay in Minutes 14.564523
           Inflight entertainment  3.920565
                      Cleanliness  2.789708
           Ease of Online booking  2.614663
                     Seat comfort  2.393453
            Inflight wifi service  2.309882
                 Inflight service  2.130857
                   Food and drink  2.114252
   Type of Travel_Personal Travel  2.078620
                        Class_Eco  1.991042
                  Online boarding  1.962958
                 Baggage handling  1.961447
Departure/Arrival time convenient  1.956572
                 On-board service  1.814315
                    Gate location  1.714485
     Customer Type_Loyal Customer  1.335237
                 Leg room service  1.327416
                   Class_Eco Plus  1.248895
                  Checkin service  1.243255

=== Satisfaction split (full data) ==

In [11]:
model_df2 = df[touchpoints + ['Arrival Delay in Minutes',
                               'Class', 'Type of Travel', 'Customer Type',
                               'satisfied_binary']].dropna()
model_df2 = pd.get_dummies(model_df2, columns=['Class', 'Type of Travel', 'Customer Type'],
                             drop_first=True)

X2 = model_df2.drop(columns=['satisfied_binary']).astype(float)
X2 = sm.add_constant(X2)
y2 = model_df2['satisfied_binary']

logit_model2 = sm.Logit(y2, X2).fit()
print(logit_model2.summary())

marginal_effects2 = logit_model2.get_margeff(at='overall')
print(marginal_effects2.summary())

Optimization terminated successfully.
         Current function value: 0.245378
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:       satisfied_binary   No. Observations:               119204
Model:                          Logit   Df Residuals:                   119184
Method:                           MLE   Df Model:                           19
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                  0.6404
Time:                        13:09:31   Log-Likelihood:                -29250.
converged:                       True   LL-Null:                       -81343.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                               -12.0825      0.095   -126

In [12]:
# 1. Top-2-box scores for each touchpoint, split by segment
def top2box_by_group(data, group_col):
    result = {}
    for group_val, group_df in data.groupby(group_col):
        scores = {}
        for col in touchpoints:
            valid = group_df[col].dropna()
            scores[col] = round((valid.isin([4, 5])).mean() * 100, 1)
        result[group_val] = scores
    return pd.DataFrame(result)

print("=== Top-2-Box % by Class ===")
print(top2box_by_group(df, 'Class'))

print("\n=== Top-2-Box % by Type of Travel ===")
print(top2box_by_group(df, 'Type of Travel'))

print("\n=== Top-2-Box % by Customer Type ===")
print(top2box_by_group(df, 'Customer Type'))

# 2. Overall satisfaction rate by segment (the headline number per group)
print("\n=== Satisfaction rate by Class ===")
print(df.groupby('Class')['satisfied_binary'].mean().round(3) * 100)

print("\n=== Satisfaction rate by Type of Travel ===")
print(df.groupby('Type of Travel')['satisfied_binary'].mean().round(3) * 100)

print("\n=== Satisfaction rate by Customer Type ===")
print(df.groupby('Customer Type')['satisfied_binary'].mean().round(3) * 100)

# 3. Satisfaction rate by Class AND Type of Travel together (the interesting cross-cut)
print("\n=== Satisfaction rate by Class x Type of Travel ===")
print((df.groupby(['Class', 'Type of Travel'])['satisfied_binary'].mean() * 100).round(1))

# 4. Delay relationship by segment — does delay hurt some segments more than others?
# Bucket delay into simple bands so the pattern is readable
df['delay_band'] = pd.cut(df['Arrival Delay in Minutes'],
                            bins=[-1, 0, 15, 60, df['Arrival Delay in Minutes'].max()],
                            labels=['No delay', '1-15 min', '16-60 min', '60+ min'])

print("\n=== Satisfaction rate by Delay Band ===")
print(df.groupby('delay_band')['satisfied_binary'].mean().round(3) * 100)

print("\n=== Satisfaction rate by Delay Band x Type of Travel ===")
print((df.groupby(['delay_band', 'Type of Travel'])['satisfied_binary'].mean() * 100).round(1))

=== Top-2-Box % by Class ===
                                   Business   Eco  Eco Plus
Inflight wifi service                  35.1  26.9      31.1
Departure/Arrival time convenient      41.7  55.5      52.1
Ease of Online booking                 40.6  26.9      30.1
Gate location                          38.7  35.0      36.0
Food and drink                         48.4  41.9      42.8
Online boarding                        68.1  34.2      37.3
Seat comfort                           68.6  44.6      45.3
Inflight entertainment                 63.5  42.6      43.4
On-board service                       63.5  42.8      40.2
Leg room service                       63.2  41.1      40.5
Baggage handling                       71.2  54.3      50.2
Checkin service                        53.8  42.9      39.9
Inflight service                       71.4  55.2      51.5
Cleanliness                            53.8  42.7      43.1

=== Top-2-Box % by Type of Travel ===
                                

/tmp/ipykernel_1618/3478124520.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('delay_band')['satisfied_binary'].mean().round(3) * 100)
/tmp/ipykernel_1618/3478124520.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print((df.groupby(['delay_band', 'Type of Travel'])['satisfied_binary'].mean() * 100).round(1))


In [13]:
# 1. Personal travel concentration checks
print("=== Type of Travel counts ===")
print(df['Type of Travel'].value_counts())
print(df['Type of Travel'].value_counts(normalize=True))

print("\n=== Type of Travel x Class ===")
print(pd.crosstab(df['Type of Travel'], df['Class'], normalize='index') * 100)

print("\n=== Age by Type of Travel ===")
print(df.groupby('Type of Travel')['Age'].describe())

# 4. Loyalty overlap check
print("\n=== Customer Type counts ===")
print(df['Customer Type'].value_counts(normalize=True))

print("\n=== Customer Type x Type of Travel ===")
print(pd.crosstab(df['Customer Type'], df['Type of Travel'], normalize='index') * 100)

# 5. Cell sizes for the cross-cuts
print("\n=== Class x Type of Travel (counts) ===")
print(pd.crosstab(df['Class'], df['Type of Travel']))

# 6. Segment group sizes
print("\n=== Group sizes ===")
print(df.groupby('Type of Travel').size())
print(df.groupby('Class').size())
print(df.groupby('Customer Type').size())

=== Type of Travel counts ===
Type of Travel
Business travel    89445
Personal Travel    40042
Name: count, dtype: int64
Type of Travel
Business travel    0.690764
Personal Travel    0.309236
Name: proportion, dtype: float64

=== Type of Travel x Class ===
Class             Business        Eco   Eco Plus
Type of Travel                                  
Business travel  66.325675  28.208396   5.465929
Personal Travel   6.655512  82.128765  11.215723

=== Age by Type of Travel ===
                   count       mean        std  min   25%   50%   75%   max
Type of Travel                                                             
Business travel  89445.0  39.883023  13.313688  7.0  29.0  40.0  50.0  85.0
Personal Travel  40042.0  38.414040  18.483052  7.0  22.0  38.0  54.0  70.0

=== Customer Type counts ===
Customer Type
Loyal Customer       0.816862
Disloyal Customer    0.183138
Name: proportion, dtype: float64

=== Customer Type x Type of Travel ===
Type of Travel     Business travel 

In [15]:
# Build the final export dataframe with everything Power BI needs
export_df = df.copy()

# Keep the delay band we already created
# (df['delay_band'] already exists from the segment analysis)

# Drop the intermediate binary column's usefulness is fine to keep — Power BI can use it directly
# Reorder isn't necessary, but drop any leftover Index-like column if present
export_cols = ['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class',
                'Flight Distance'] + touchpoints + [
                'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'delay_band',
                'satisfaction', 'satisfied_binary']

export_df = export_df[export_cols]

print(f"Exporting {len(export_df)} rows, {len(export_df.columns)} columns")
print(export_df.dtypes)

export_df.to_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/processed/cleaned_satisfaction_data.csv', index=False)
print("Saved to Drive.")

Exporting 129487 rows, 25 columns
Gender                                 object
Customer Type                          object
Age                                     int64
Type of Travel                         object
Class                                  object
Flight Distance                         int64
Inflight wifi service                 float64
Departure/Arrival time convenient     float64
Ease of Online booking                float64
Gate location                         float64
Food and drink                        float64
Online boarding                       float64
Seat comfort                          float64
Inflight entertainment                float64
On-board service                      float64
Leg room service                      float64
Baggage handling                        int64
Checkin service                       float64
Inflight service                      float64
Cleanliness                           float64
Departure Delay in Minutes              int64


In [16]:
# Export the marginal effects as a small, Power-BI-ready table
mfx_export = marginal_effects2.summary_frame().reset_index()
mfx_export.columns = ['Touchpoint', 'Marginal Effect (pp)', 'Std Err', 'z', 'P>|z|', 'CI Lower', 'CI Upper']

# Keep only the 14 touchpoints (drop the controls — Class, Type of Travel, Customer Type, Arrival Delay —
# since those aren't touchpoints for the "Drivers" bar chart)
mfx_export = mfx_export[mfx_export['Touchpoint'].isin(touchpoints)].copy()

# Convert to percentage points for readability, and sort by effect size
mfx_export['Marginal Effect (pp)'] = (mfx_export['Marginal Effect (pp)'] * 100).round(2)
mfx_export = mfx_export.sort_values('Marginal Effect (pp)', ascending=False)

print(mfx_export[['Touchpoint', 'Marginal Effect (pp)']])

mfx_export.to_csv('/content/drive/MyDrive/mlas-cx-dashboard/data/processed/driver_effects.csv', index=False)
print("Saved to Drive.")

                           Touchpoint  Marginal Effect (pp)
5                     Online boarding                  6.75
0               Inflight wifi service                  5.76
2              Ease of Online booking                  2.85
11                    Checkin service                  2.72
8                    On-board service                  2.69
9                    Leg room service                  2.37
13                        Cleanliness                  2.00
10                   Baggage handling                  1.23
12                   Inflight service                  1.23
7              Inflight entertainment                  0.60
6                        Seat comfort                  0.18
4                      Food and drink                 -0.35
3                       Gate location                 -1.91
1   Departure/Arrival time convenient                 -2.53
Saved to Drive.
